# 05 - Position Scoring & Comparison

This notebook calculates composite position scores for players based on PCA-transformed statistics.

**Prerequisites**: Complete all previous notebooks or run the full pipeline with `scripts/pipeline.py`

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import modules
from src.utils import DataManager

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("✓ Modules imported successfully")

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


## Setup & Data Loading

In [ ]:
# Initialize utilities
manager = DataManager()

# Load player profiles
try:
    df_players = manager.load_pickle('data/processed/players_profiles.pkl')
    print(f"✓ Loaded player profiles: {df_players.shape}")
except FileNotFoundError:
    print("⚠ Player profiles not found")
    df_players = None

# Load position scores (from script 06)
try:
    df_scores = pd.read_csv('data/processed/player_position_scores.csv')
    print(f"✓ Loaded position scores: {df_scores.shape}")
except FileNotFoundError:
    print("⚠ Position scores not available yet")
    df_scores = None


## Step 1: Examine Scoring Results

In [ ]:
# Join with player details for final export
if df_scores is not None and df_players is not None:
    # Merge scores with player details
    df_export = df_scores.merge(
        df_players[['wyId', 'name', 'team_name', 'role', 'birth_date']], 
        on='wyId',
        how='left'
    )
    
    print(f"✓ Export dataset shape: {df_export.shape}")
    print(f"\nExport preview (top 5):")
    print(df_export.head())
    
    # Save for external use
    export_path = 'data/exports/player_scoring_results.csv'
    df_export.to_csv(export_path, index=False)
    print(f"\n✓ Saved to {export_path}")


## Step 4: Export Results

In [ ]:
# Visualize score distributions
if df_scores is not None:
    position_cols = [col for col in df_scores.columns if 'score' in col.lower()]
    
    fig, axes = plt.subplots(len(position_cols), 1, figsize=(12, 3*len(position_cols)))
    if len(position_cols) == 1:
        axes = [axes]
    
    for ax, pos_col in zip(axes, position_cols):
        ax.hist(df_scores[pos_col].dropna(), bins=30, alpha=0.7, color='steelblue')
        ax.set_xlabel(pos_col)
        ax.set_ylabel('Number of Players')
        ax.set_title(f'Distribution of {pos_col}')
        ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Visualization complete")


## Step 3: Score Distribution Visualization

In [ ]:
# Find top players for each position
if df_scores is not None:
    # Position columns (assuming format: defender_score, midfielder_score, forward_score)
    position_cols = [col for col in df_scores.columns if 'score' in col.lower()]
    
    print(f"✓ Position Scores Columns: {position_cols}\n")
    
    for pos_col in position_cols[:3]:  # Show top 3 positions
        if pos_col in df_scores.columns:
            top_players = df_scores.nlargest(10, pos_col)[['wyId', pos_col]]
            print(f"\nTop 10 Players - {pos_col}:")
            print(top_players.to_string())


## Step 2: Top Players by Position (if scores available)

In [ ]:
# Examine scores dataframe
if df_scores is not None:
    print("✓ Position Scores DataFrame:")
    print(f"  Shape: {df_scores.shape}")
    print(f"  Columns: {df_scores.columns.tolist()}")
    print(f"\nFirst few rows:")
    print(df_scores.head(10))
    print(f"\nScore statistics:")
    print(df_scores.describe())


## Summary

This concludes the 5-notebook series:
1. ✓ **Data Exploration**: Fetch competitions and teams
2. ✓ **Player Analysis**: Analyze careers and player distribution
3. ✓ **Advanced Stats**: Load and clean match statistics
4. ✓ **Clustering**: Apply PCA for dimensionality reduction
5. ✓ **Position Scoring**: Calculate composite position scores

**Next Steps**:
- Use these results for scouting recommendations
- Integrate with other data sources (market value, contract info, etc.)
- Build a web dashboard to visualize player profiles
- Create player comparison reports for scouts/coaches